<a href="https://colab.research.google.com/github/yiguo0408-tech/AML/blob/main/AML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install -q kagglehub
# !pip install -q torch torchvision torchaudio
# !pip install -q torch-geometric
# !pip install -q scikit-learn pandas numpy matplotlib networkx
# !pip install -q xgboost

In [ ]:
import os
import random
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# PyTorch / PyG
import torch
import torch.nn as nn
import torch.nn.functional as F

!pip install -q torch-geometric # Added this line to install torch_geometric

from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, remove_self_loops, add_self_loops
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv

# Scikit-learn
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Optional XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.2 MB/s eta 0:00:00


In [ ]:
import kagglehub
import os
import pandas as pd

# Loading the files

# 1. Download dataset
path = kagglehub.dataset_download("ellipticco/elliptic-data-set")

# 2. Define file paths
features_path = os.path.join(path, "elliptic_bitcoin_dataset", "elliptic_txs_features.csv")
classes_path  = os.path.join(path, "elliptic_bitcoin_dataset", "elliptic_txs_classes.csv")
edges_path    = os.path.join(path, "elliptic_bitcoin_dataset", "elliptic_txs_edgelist.csv")

# 3. Load files
features = pd.read_csv(features_path, header=None)
classes = pd.read_csv(classes_path)
edges = pd.read_csv(edges_path)

Using Colab cache for faster access to the 'elliptic-data-set' dataset.


In [ ]:
feature_cols = ["txId", "time_step"] + [f"feat_{i}" for i in range(features.shape[1] - 2)]
features.columns = feature_cols

classes.columns = ["txId", "class"]
edges.columns = ["txId1", "txId2"]

In [ ]:
# Merge the features and labels data

# Rename columns for clarity
feature_cols = ["txId", "time_step"] + [f"feat_{i}" for i in range(features.shape[1] - 2)]
features.columns = feature_cols
classes.columns = ["txId", "class"]
edges.columns = ["txId1", "txId2"]

# Standardize ids as strings
features["txId"] = features["txId"].astype(str)
classes["txId"] = classes["txId"].astype(str)
edges["txId1"] = edges["txId1"].astype(str)
edges["txId2"] = edges["txId2"].astype(str)

# Map labels:
# 1 -> illicit, 2 -> licit, unknown -> -1
class_map = {"1": 1, "2": 0, "unknown": -1}
classes["label"] = classes["class"].map(class_map)

# Merge
df = features.merge(classes[["txId", "label"]], on="txId", how="left")
df["label"] = df["label"].fillna(-1).astype(int)

print("Merged df shape:", df.shape)
print(df["label"].value_counts().sort_index())
df.head()

Merged df shape: (203769, 168)
label
-1    157205
 0     42019
 1      4545
Name: count, dtype: int64


,txId,time_step,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,...,feat_156,feat_157,feat_158,feat_159,feat_160,feat_161,feat_162,feat_163,feat_164,label
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,-1
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,-1
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792,-1
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792,0
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117,-1


In [ ]:
# Merge the time and edge data / graph cleanup

# 1. Remove unknown nodes first
df = df[df["label"] != -1].copy()

# 2. Remove duplicated rows
df = df.drop_duplicates(subset=["txId"]).reset_index(drop=True)
edges = edges.drop_duplicates().reset_index(drop=True)

# 3. Keep only edges whose endpoints exist in df
valid_ids = set(df["txId"])
edges = edges[edges["txId1"].isin(valid_ids) & edges["txId2"].isin(valid_ids)].copy()

print("Cleaned nodes (without unknown):", len(df))
print("Cleaned edges:", len(edges))
print("Time range:", df["time_step"].min(), "to", df["time_step"].max())
print(df["label"].value_counts(dropna=False))

Cleaned nodes (without unknown): 46564
Cleaned edges: 36624
Time range: 1 to 49
label
0    42019
1     4545
Name: count, dtype: int64


In [ ]:
TRAIN_TIME_MAX = 34
VAL_TIME_MAX = 41

train_mask = (df["time_step"] <= TRAIN_TIME_MAX)
val_mask   = (df["time_step"] > TRAIN_TIME_MAX) & (df["time_step"] <= VAL_TIME_MAX)
test_mask  = (df["time_step"] > VAL_TIME_MAX)

print("Train nodes:", train_mask.sum())
print("Val nodes:", val_mask.sum())
print("Test nodes:", test_mask.sum())

print("\nTrain label ratio:")
print(df.loc[train_mask, "label"].value_counts(normalize=True))
print("\nVal label ratio:")
print(df.loc[val_mask, "label"].value_counts(normalize=True))
print("\nTest label ratio:")
print(df.loc[test_mask, "label"].value_counts(normalize=True))

Train nodes: 29894
Val nodes: 7829
Test nodes: 8841

Train label ratio:
label
0    0.884191
1    0.115809
Name: proportion, dtype: float64

Val label ratio:
label
0    0.913782
1    0.086218
Name: proportion, dtype: float64

Test label ratio:
label
0    0.953851
1    0.046149
Name: proportion, dtype: float64


In [ ]:
# Feature preprocessing and graph construction

# ----- feature preprocessing -----
feat_cols = [c for c in df.columns if c.startswith("feat_")]
X = df[feat_cols].copy()

# Fill missing values using train medians
train_medians = X.loc[train_mask].median()
X = X.fillna(train_medians)

# Remove zero-variance columns based on train split
train_std = X.loc[train_mask].std()
keep_cols = train_std[train_std > 0].index.tolist()
X = X[keep_cols]

# Standardize using train split only
scaler = StandardScaler()
scaler.fit(X.loc[train_mask].values)

X_scaled = scaler.transform(X.values)
y = df["label"].values

# ----- build base graph -----
id2idx = {txid: idx for idx, txid in enumerate(df["txId"].tolist())}
src = edges["txId1"].map(id2idx).values
dst = edges["txId2"].map(id2idx).values

base_edges = np.vstack([src, dst]).T
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
edge_index = to_undirected(edge_index)
edge_index, _ = remove_self_loops(edge_index)
edge_index, _ = add_self_loops(edge_index, num_nodes=len(df))

# ----- PyG data object -----
x_tensor = torch.tensor(X_scaled, dtype=torch.float)
y_tensor = torch.tensor(y, dtype=torch.long)

data = Data(x=x_tensor, edge_index=edge_index, y=y_tensor)
data.train_mask = torch.tensor(train_mask.values if hasattr(train_mask, "values") else train_mask, dtype=torch.bool)
data.val_mask   = torch.tensor(val_mask.values if hasattr(val_mask, "values") else val_mask, dtype=torch.bool)
data.test_mask  = torch.tensor(test_mask.values if hasattr(test_mask, "values") else test_mask, dtype=torch.bool)

# time index for later temporal models
time_values = df["time_step"].values.astype(int)
unique_times = np.sort(np.unique(time_values))
time_to_idx = {t: i for i, t in enumerate(unique_times)}
time_idx = np.array([time_to_idx[t] for t in time_values], dtype=np.int64)
num_time_steps = len(unique_times)
data.time_idx = torch.tensor(time_idx, dtype=torch.long)

print("X_scaled shape:", X_scaled.shape)
print("PyG data:", data)

X_scaled shape: (46564, 165)
PyG data: Data(x=[46564, 165], edge_index=[2, 119812], y=[46564], train_mask=[46564], val_mask=[46564], test_mask=[46564], time_idx=[46564])


In [ ]:
# Phase 7.0 - 准备传统机器学习模型的训练 / 验证 / 测试数据

# Re-generate masks based on the filtered df (length 46564).
# The 'df' variable in the kernel state already reflects the filtered data (46564 rows)
# where 'label != -1' condition is implicitly handled by the filtering in cell ygWZnntRfp_L.
# Assuming TRAIN_TIME_MAX and VAL_TIME_MAX are available from previous cells.
train_mask = (df["time_step"] <= TRAIN_TIME_MAX)
val_mask   = (df["time_step"] > TRAIN_TIME_MAX) & (df["time_step"] <= VAL_TIME_MAX)
test_mask  = (df["time_step"] > VAL_TIME_MAX)

# 训练集
X_train = X_scaled[train_mask]
y_train = y[train_mask]

# 验证集
X_val = X_scaled[val_mask]
y_val = y[val_mask]

# 测试集
X_test = X_scaled[test_mask]
y_test = y[test_mask]

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

print("Train label distribution:")
print(pd.Series(y_train).value_counts().sort_index())

X_train shape: (29894, 165)
X_val shape: (7829, 165)
X_test shape: (8841, 165)
Train label distribution:
0    26432
1     3462
Name: count, dtype: int64


In [ ]:
def evaluate_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob)

    illicit_precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    illicit_recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    illicit_f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

    licit_recall = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    g_mean = np.sqrt(illicit_recall * licit_recall)

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "ROC-AUC": roc_auc,
        "Illicit Precision": illicit_precision,
        "Illicit Recall": illicit_recall,
        "Illicit F1": illicit_f1,
        "G-Mean": g_mean
    }


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from IPython.display import display

# 用字典保存所有模型结果
baseline_results = {}

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=SEED
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=SEED,
        n_jobs=-1
    ),
    "Support Vector Machine": SVC(kernel="rbf", probability=False, random_state=SEED),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        eval_metric="logloss",
        random_state=SEED
    )
}

# Loop through models, train, predict and evaluate
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        # For SVM without probability=True or other models without predict_proba
        y_prob = model.decision_function(X_test)
        # Scale to [0, 1] if needed, or use a fixed threshold
        # For now, let's assume decision_function's sign indicates class
        # and for metrics like ROC-AUC, raw scores are often fine

    metrics = evaluate_metrics(y_test, y_prob)
    baseline_results[name] = metrics
    print(f"{name} metrics: {metrics}")


# --------------------------------------------
# 5. 转成结果表格
# --------------------------------------------
baseline_results_df = pd.DataFrame(baseline_results).T

# 按 Illicit F1 从高到低排序
baseline_results_df = baseline_results_df.sort_values(by="Illicit F1", ascending=False)

print("===== Baseline Model Comparison (Test Set) ====")
display(baseline_results_df)

Training Logistic Regression...
Logistic Regression metrics: {'Accuracy': 0.8695848885872639, 'Precision': 0.9391393124140498, 'Recall': 0.8695848885872639, 'F1-Score': 0.8982747172674096, 'ROC-AUC': np.float64(0.8497080796032395), 'Illicit Precision': 0.18618365627632688, 'Illicit Recall': 0.5416666666666666, 'Illicit F1': 0.27711598746081506, 'G-Mean': np.float64(0.6925451318396283)}
Training Random Forest...
Random Forest metrics: {'Accuracy': 0.9727406401990725, 'Precision': 0.9707928119786465, 'Recall': 0.9727406401990725, 'F1-Score': 0.9687280317522164, 'ROC-AUC': np.float64(0.8178377778242805), 'Illicit Precision': 0.8847926267281107, 'Illicit Recall': 0.47058823529411764, 'Illicit F1': 0.6144, 'G-Mean': np.float64(0.6849767556200035)}
Training Support Vector Machine...
Support Vector Machine metrics: {'Accuracy': 0.959393733740527, 'Precision': 0.9520689748787389, 'Recall': 0.959393733740527, 'F1-Score': 0.953901673366377, 'ROC-AUC': np.float64(0.7972195192555857), 'Illicit Pre

,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Illicit Precision,Illicit Recall,Illicit F1,G-Mean
Random Forest,0.972741,0.970793,0.972741,0.968728,0.817838,0.884793,0.470588,0.614400,0.684977
XGBoost,0.968329,0.964537,0.968329,0.965070,0.854888,0.742424,0.480392,0.583333,0.690303
Support Vector Machine,0.959394,0.952069,0.959394,0.953902,0.797220,0.606987,0.340686,0.436421,0.580560
Logistic Regression,0.869585,0.939139,0.869585,0.898275,0.849708,0.186184,0.541667,0.277116,0.692545


In [ ]:
# ============================================
# Phase 6.1 - 定义图神经网络 baseline 模型
# ============================================

from torch_geometric.nn import GCNConv, GATConv, GINConv, SAGEConv

# ------------------------------
# 1. GCN
# ------------------------------
class GCNNet(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = nn.Linear(hidden_dim, 2)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.lin(x)
        return x


# ------------------------------
# 2. GAT
# ------------------------------
class GATNet(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, heads=4, dropout=0.3):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_dim * heads, hidden_dim, heads=1, concat=True, dropout=dropout)
        self.lin = nn.Linear(hidden_dim, 2)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.lin(x)
        return x


# ------------------------------
# 3. GIN
# ------------------------------
class GINNet(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, dropout=0.3):
        super().__init__()

        mlp1 = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        mlp2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.conv1 = GINConv(mlp1)
        self.conv2 = GINConv(mlp2)
        self.lin = nn.Linear(hidden_dim, 2)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.lin(x)
        return x


# ------------------------------
# 4. GraphSAGE
# ------------------------------
class GraphSAGENet(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.lin = nn.Linear(hidden_dim, 2)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.lin(x)
        return x

In [ ]:
# ============================================
# Phase 6.2 - 定义统一训练函数
# ============================================

def train_gnn_baseline(model, data, epochs=100, lr=1e-3, weight_decay=5e-4):
    model = model.to(DEVICE)
    data = data.to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()

        out = model(data.x, data.edge_index)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])

        loss.backward()
        optimizer.step()

    return model

In [ ]:
# ============================================
# Phase 6.3 - 提取图模型测试集概率
# ============================================

def get_gnn_test_prob(model, data):
    """
    输出图模型在测试集上的 illicit 概率
    """
    model.eval()
    data = data.to(DEVICE)

    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        prob = F.softmax(logits, dim=1)[:, 1].cpu().numpy()

    y_test_local = data.y[data.test_mask].cpu().numpy()
    test_prob = prob[data.test_mask.cpu().numpy()]

    return y_test_local, test_prob

In [ ]:
# ============================================
# Phase 6.4 - 图神经网络 baseline comparison
# ============================================

gnn_results = {}

# --- FIX: Re-assign masks for the data object to match current df size ---
# The train_mask, val_mask, test_mask variables are now correctly sized (length 46564)
# after cell FN5Uh6RjiXav. We need to update the PyG data object with these.
data.train_mask = torch.tensor(train_mask.values, dtype=torch.bool)
data.val_mask   = torch.tensor(val_mask.values, dtype=torch.bool)
data.test_mask  = torch.tensor(test_mask.values, dtype=torch.bool)
# -----------------------------------------------------------------------

# ------------------------------
# 1. GCN
# ------------------------------
gcn_model = GCNNet(in_dim=data.x.shape[1], hidden_dim=64, dropout=0.3)
gcn_model = train_gnn_baseline(gcn_model, data)

y_test_gcn, gcn_test_prob = get_gnn_test_prob(gcn_model, data)
gnn_results["GCN"] = evaluate_metrics(y_test_gcn, gcn_test_prob)

# ------------------------------
# 2. GAT
# ------------------------------
gat_model = GATNet(in_dim=data.x.shape[1], hidden_dim=64, heads=4, dropout=0.3)
gat_model = train_gnn_baseline(gat_model, data)

y_test_gat, gat_test_prob = get_gnn_test_prob(gat_model, data)
gnn_results["GAT"] = evaluate_metrics(y_test_gat, gat_test_prob)

# ------------------------------
# 3. GIN
# ------------------------------
gin_model = GINNet(in_dim=data.x.shape[1], hidden_dim=64, dropout=0.3)
gin_model = train_gnn_baseline(gin_model, data)

y_test_gin, gin_test_prob = get_gnn_test_prob(gin_model, data)
gnn_results["GIN"] = evaluate_metrics(y_test_gin, gin_test_prob)

# ------------------------------
# 4. GraphSAGE
# ------------------------------
sage_model = GraphSAGENet(in_dim=data.x.shape[1], hidden_dim=64, dropout=0.3)
sage_model = train_gnn_baseline(sage_model, data)

y_test_sage, sage_test_prob = get_gnn_test_prob(sage_model, data)
gnn_results["GraphSAGE"] = evaluate_metrics(y_test_sage, sage_test_prob)

# ------------------------------
# 5. 转成表格
# ------------------------------
gnn_results_df = pd.DataFrame(gnn_results).T
gnn_results_df = gnn_results_df.sort_values(by="Illicit F1", ascending=False)

print("===== GNN Baseline Comparison (Test Set) ====")
display(gnn_results_df)

# ============================================
# Phase 6.5 / 7.5 - 合并传统模型与图模型结果
# ============================================

all_results_df = pd.concat([baseline_results_df, gnn_results_df], axis=0)
all_results_df = all_results_df.sort_values(by="Illicit F1", ascending=False)

print("===== Overall Baseline Comparison ====")
display(all_results_df)

===== GNN Baseline Comparison (Test Set) ====


,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Illicit Precision,Illicit Recall,Illicit F1,G-Mean
GAT,0.955661,0.952304,0.955661,0.953797,0.829716,0.523392,0.438725,0.477333,0.655931
GraphSAGE,0.965389,0.960687,0.965389,0.958748,0.825179,0.789773,0.340686,0.476027,0.582401
GCN,0.966746,0.965843,0.966746,0.958424,0.805754,0.938462,0.299020,0.453532,0.546567
GIN,0.957810,0.949883,0.957810,0.952104,0.758719,0.576419,0.323529,0.414443,0.565516


===== Overall Baseline Comparison ====


,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Illicit Precision,Illicit Recall,Illicit F1,G-Mean
Random Forest,0.972741,0.970793,0.972741,0.968728,0.817838,0.884793,0.470588,0.614400,0.684977
XGBoost,0.968329,0.964537,0.968329,0.965070,0.854888,0.742424,0.480392,0.583333,0.690303
GAT,0.955661,0.952304,0.955661,0.953797,0.829716,0.523392,0.438725,0.477333,0.655931
GraphSAGE,0.965389,0.960687,0.965389,0.958748,0.825179,0.789773,0.340686,0.476027,0.582401
GCN,0.966746,0.965843,0.966746,0.958424,0.805754,0.938462,0.299020,0.453532,0.546567
Support Vector Machine,0.959394,0.952069,0.959394,0.953902,0.797220,0.606987,0.340686,0.436421,0.580560
GIN,0.957810,0.949883,0.957810,0.952104,0.758719,0.576419,0.323529,0.414443,0.565516
Logistic Regression,0.869585,0.939139,0.869585,0.898275,0.849708,0.186184,0.541667,0.277116,0.692545
